# Stage 2: Fingerprinty i przygotowanie danych treningowych

Ten notatnik:
1. Wczytuje dane po etapie 1.
2. Buduje fingerprinty (`ECFP`, `MACCS`).
3. Tworzy split oparty o scaffolds (Murcko).
4. Tworzy macierze hierarchii klas z pliku OBO.
5. Zapisuje artefakty do dalszego treningu modelu.

In [1]:
from pathlib import Path
import json
import re
from typing import Dict, List, Set

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data/chebi_dataset_train_stage1.parquet"
OBO_PATH = PROJECT_ROOT / "extras/chebi_classes.obo"
DEFS_PATH = PROJECT_ROOT / "extras/chebi_class_definitions.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "data/stage2_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_RE = re.compile(r"^class_(\d+)$")

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Nieobsługiwany format: {suffix}")

def find_class_columns(columns: List[str]) -> List[str]:
    pairs = []
    for c in columns:
        m = CLASS_RE.match(c)
        if m:
            pairs.append((int(m.group(1)), c))
    pairs.sort(key=lambda x: x[0])
    return [c for _, c in pairs]

def parse_obo_parents(obo_path: Path) -> Dict[str, Set[str]]:
    parents: Dict[str, Set[str]] = {}
    current_id = None
    with obo_path.open("r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.strip()
            if line == "[Term]":
                current_id = None
                continue
            if line.startswith("id: "):
                current_id = line[4:].strip()
                parents.setdefault(current_id, set())
                continue
            if line.startswith("is_a: ") and current_id:
                parent_id = line[6:].split("!")[0].strip()
                if parent_id:
                    parents[current_id].add(parent_id)
    return parents

def compute_ancestors(class_cols: List[str], parent_map: Dict[str, Set[str]]) -> Dict[str, Set[str]]:
    allowed = set(class_cols)
    memo: Dict[str, Set[str]] = {}

    def dfs(node: str, stack: Set[str] | None = None) -> Set[str]:
        if node in memo:
            return memo[node]
        if stack is None:
            stack = set()
        if node in stack:
            return set()
        stack = set(stack)
        stack.add(node)

        direct = {p for p in parent_map.get(node, set()) if p in allowed}
        out = set(direct)
        for p in direct:
            out.update(dfs(p, stack))
        memo[node] = out
        return out

    for cls in class_cols:
        dfs(cls)
    return memo

def scaffold_from_smiles(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


In [3]:
df = read_table(DATA_PATH)
class_cols = find_class_columns(df.columns.tolist())

required = {"mol_id", "SMILES"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Brakuje kolumn wymaganych: {missing_required}")

if not class_cols:
    raise ValueError("Nie znaleziono kolumn class_i")

if "is_valid_smiles" in df.columns:
    df = df[df["is_valid_smiles"].astype(bool)].copy()

smiles_col = "canonical_smiles" if "canonical_smiles" in df.columns else "SMILES"
df = df[df[smiles_col].notna()].copy()
df[smiles_col] = df[smiles_col].astype(str)

for c in class_cols:
    df[c] = df[c].astype(bool)

print("Liczba rekordów:", len(df))
print("Liczba klas:", len(class_cols))
print("Kolumna SMILES do cech:", smiles_col)
df[["mol_id", smiles_col] + class_cols[:5]].head(3)

Liczba rekordów: 33631
Liczba klas: 500
Kolumna SMILES do cech: canonical_smiles


,mol_id,canonical_smiles,class_0,class_1,class_2,class_3,class_4
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,True,True,True,True,True
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,True,True,True,True,True
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,True,True,True,True,True


In [4]:
df.head()

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_493,class_494,class_495,class_496,class_497,class_498,class_499,is_valid_smiles,canonical_smiles,inchikey
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,True,CCCCC/C=C\CCCCCCCC(=O)O,DJCQJZKZUCHHAL-SREVYHEPSA-N
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,True,Cc1cc2cc(O)cc(O)c2c(C)n1,NFCOBHKSUZDTLE-UHFFFAOYSA-N
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,True,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,JUGXQEJPWDYOJV-YSQMORBQSA-N
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,True,True,True,True,True,True,True,False,...,False,False,False,False,False,False,False,True,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,CNMRNUFFAOLBLH-UHFFFAOYSA-N
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,False,True,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,UXWZMQPNSYWAHX-DMELVVMMSA-N


In [5]:
# Fingerprinty
smiles = df[smiles_col].tolist()

ecfp = ECFPFingerprint(fp_size=2048, radius=2, include_chirality=True, n_jobs=-1)
maccs = MACCSFingerprint(n_jobs=-1)

X_ecfp = ecfp.transform(smiles).astype(np.uint8)
X_maccs = maccs.transform(smiles).astype(np.uint8)
Y = df[class_cols].astype(np.uint8).to_numpy()

print("X_ecfp:", X_ecfp.shape, X_ecfp.dtype)
print("X_maccs:", X_maccs.shape, X_maccs.dtype)
print("Y:", Y.shape, Y.dtype)

X_ecfp: (33631, 2048) uint8
X_maccs: (33631, 166) uint8
Y: (33631, 500) uint8


In [6]:
# Hierarchia klas (macierze parent/ancestor)
parent_map = parse_obo_parents(OBO_PATH)
anc_map = compute_ancestors(class_cols, parent_map)

class_idx = {c: i for i, c in enumerate(class_cols)}
C = len(class_cols)
M_parent = np.zeros((C, C), dtype=np.uint8)
M_ancestor = np.zeros((C, C), dtype=np.uint8)

for child, parents in parent_map.items():
    if child not in class_idx:
        continue
    i = class_idx[child]
    for p in parents:
        if p in class_idx:
            M_parent[i, class_idx[p]] = 1

for child, ancestors in anc_map.items():
    if child not in class_idx:
        continue
    i = class_idx[child]
    for a in ancestors:
        if a in class_idx:
            M_ancestor[i, class_idx[a]] = 1

print("M_parent:", M_parent.shape, "edges:", int(M_parent.sum()))
print("M_ancestor:", M_ancestor.shape, "edges:", int(M_ancestor.sum()))

M_parent: (500, 500) edges: 748
M_ancestor: (500, 500) edges: 8610


In [7]:
# Split train/valid po scaffoldach (mniejszy leakage)
scaffolds = np.array([scaffold_from_smiles(s) for s in smiles])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx = np.arange(len(df))
train_idx, valid_idx = next(gss.split(idx, groups=scaffolds))

print("Train:", len(train_idx), "Valid:", len(valid_idx))
print("Unikalne scaffolds (train):", len(set(scaffolds[train_idx])))
print("Unikalne scaffolds (valid):", len(set(scaffolds[valid_idx])))

[15:44:26] WARNING: not removing hydrogen atom without neighbors
[15:44:26] WARNING: not removing hydrogen atom without neighbors
[15:44:26] WARNING: not removing hydrogen atom without neighbors
[15:44:26] WARNING: not removing hydrogen atom without neighbors
[15:44:27] WARNING: not removing hydrogen atom without neighbors
[15:44:27] WARNING: not removing hydrogen atom without neighbors
[15:44:27] WARNING: not removing hydrogen atom without neighbors
[15:44:27] WARNING: not removing hydrogen atom without neighbors
[15:44:27] WARNING: not removing hydrogen atom without neighbors
[15:44:27] Unusual charge on atom 0 number of radical electrons set to zero
[15:44:28] WARNING: not removing hydrogen atom without neighbors
[15:44:28] WARNING: not removing hydrogen atom without neighbors
[15:44:28] WARNING: not removing hydrogen atom without neighbors
[15:44:28] WARNING: not removing hydrogen atom without neighbors
[15:44:28] WARNING: not removing hydrogen atom without neighbors
[15:44:28] WAR

Train: 20682 Valid: 12949
Unikalne scaffolds (train): 5850
Unikalne scaffolds (valid): 1463


In [8]:
# Zapis artefaktów
artifact_npz = ARTIFACTS_DIR / "stage2_fingerprints.npz"
artifact_meta = ARTIFACTS_DIR / "stage2_fingerprints_meta.json"
artifact_index = ARTIFACTS_DIR / "stage2_row_index.parquet"

np.savez_compressed(
    artifact_npz,
    X_ecfp=X_ecfp,
    X_maccs=X_maccs,
    Y=Y,
    train_idx=train_idx,
    valid_idx=valid_idx,
    M_parent=M_parent,
    M_ancestor=M_ancestor,
)

index_cols = ["mol_id", smiles_col]
if "inchikey" in df.columns:
    index_cols.append("inchikey")
df[index_cols].reset_index(drop=True).to_parquet(artifact_index, index=False)

class_names = {}
if DEFS_PATH.exists():
    defs = pd.read_csv(DEFS_PATH)
    if {"chebi_id", "name"}.issubset(defs.columns):
        class_names = dict(zip(defs["chebi_id"], defs["name"]))

meta = {
    "data_path": str(DATA_PATH),
    "n_rows": int(len(df)),
    "n_classes": int(len(class_cols)),
    "smiles_column": smiles_col,
    "ecfp_shape": list(X_ecfp.shape),
    "maccs_shape": list(X_maccs.shape),
    "target_shape": list(Y.shape),
    "train_size": int(len(train_idx)),
    "valid_size": int(len(valid_idx)),
    "class_columns": class_cols,
    "class_names": {c: class_names.get(c, "") for c in class_cols},
    "files": {
        "npz": str(artifact_npz),
        "meta_json": str(artifact_meta),
        "row_index": str(artifact_index),
    },
}

with artifact_meta.open("w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Zapisano:")
print("-", artifact_npz)
print("-", artifact_meta)
print("-", artifact_index)

Zapisano:
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints.npz
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints_meta.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_row_index.parquet


In [9]:
# Szybki podgląd częstości klas (top-10)
freq = pd.Series(Y.mean(axis=0), index=class_cols).sort_values(ascending=False)
top10 = freq.head(10).to_frame("positive_fraction")
top10

,positive_fraction
class_0,1.000000
class_1,0.994172
class_2,0.974577
class_3,0.962951
class_4,0.923136
class_5,0.921233
class_6,0.823585
class_7,0.743956
class_8,0.720972
class_9,0.695846
